# Citation Audit Workflow

Citation audit is the pre-writing check that asks whether your registry, notes, claims, evidence locations, BibTeX keys, and themes are internally consistent.

This notebook uses the synthetic example corpus only. It does not decide whether any scientific claim is true.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "paper_workbench").exists():
    root = root.parent
sys.path.insert(0, str(root))

registry_path = root / "data" / "registries" / "example_papers.csv"
bibtex_path = root / "data" / "bibtex" / "example_library.bib"
notes_dir = root / "data" / "notes"
themes_path = root / "data" / "examples" / "themes.json"

## Load Audit Inputs

The audit needs all major local inputs: registry papers, parsed notes, extracted claims, BibTeX entries, and theme definitions.

In [ ]:
from paper_workbench.bibtex import parse_bibtex_file
from paper_workbench.claims import collect_claims, collect_notes
from paper_workbench.registry import load_registry
from paper_workbench.tags import load_themes

papers = load_registry(registry_path)
notes = collect_notes(notes_dir)
claims = collect_claims(notes_dir)
entries = parse_bibtex_file(bibtex_path)
themes = load_themes(themes_path)

{
    "papers": len(papers),
    "notes": len(notes),
    "claims": len(claims),
    "bibtex_entries": len(entries),
    "themes": len(themes),
}

## Run the Citation Audit

Findings are grouped by severity and code so you can triage release-blocking citation problems before drafting.

In [ ]:
from collections import Counter
from paper_workbench.audit import citation_audit

findings = citation_audit(papers, notes, claims, entries, themes, root=root)
{
    "severity_counts": Counter(finding.severity for finding in findings),
    "code_counts": Counter(finding.code for finding in findings),
}

## Inspect Actionable Findings

The important audit behavior is not the count alone. Read the message and suggestion before deciding whether to update a note, registry row, BibTeX entry, or theme.

In [ ]:
[(finding.severity, finding.code, finding.paper_id or finding.theme, finding.suggestion) for finding in findings[:8]]

## Preview the Markdown Report

The CLI writes the same content to Markdown. The notebook keeps it in memory so this example does not overwrite repository reports.

In [ ]:
from paper_workbench.reporting import citation_audit_report

report_markdown = citation_audit_report(findings)
report_markdown.splitlines()[:12]

## Key Takeaways

- Citation audit checks completeness of user-tracked evidence; it does not validate scientific truth.
- Missing evidence locations and missing notes should be fixed before using claims in prose.
- Registry and BibTeX link problems are citation-management risks.
- Under-supported themes indicate where follow-up reading or note cleanup is needed.